In [8]:
import os
import sys
import json
sys.path.insert(0, '/data2/polyakov/instruction_tuning/ToolBench')
sys.path.insert(0, '/data2/polyakov/instruction_tuning/ToolBench/toolbench')
sys.path.insert(0, '/data2/polyakov/instruction_tuning/ToolBench/toolbench/inference')
os.environ["CUDA_VISIBLE_DEVICES"]="0, 1"

In [9]:
from toolbench.model.model_adapter import get_conversation_template

In [17]:
with open('data/self_correction/train_data_cleaned_g1_callable.json', 'r') as f:
    all_callable_examples = json.load(f)

template = "tool-llama-single-round"
roles = {"system": conv.roles[0], "user": conv.roles[1], "function": conv.roles[2], "assistant": conv.roles[3]}

conv = get_conversation_template(template)
source = all_callable_examples[0]['conversations']
for j, sentence in enumerate(source):
        role = roles[sentence["from"]]
        conv.append_message(role, sentence["value"])
prompt = conv.get_prompt()

In [18]:
print(prompt)

System: You are AutoGPT, you can use many tools(functions) to do the following task.
First I will give you the task description, and your task start.
At each step, you need to give your thought to analyze the status now and what to do next, with a function call to actually excute your step. Your output should follow this format:
Thought:
Action
Action Input:

After the call, you will get the call result, and you are now in a new state.
Then you will analyze your status now, then decide what to do next...
After many (Thought-call) pairs, you finally perform the task, then you can give your finial answer.
Remember: 
1.the state change is irreversible, you can't go back to one of the former state, if you want to restart the task, say "I give up and restart".
2.All the thought is short, at most in 5 sentence.
3.You can do more then one trys, so if your plan is to continusly try some conditions, you can do one of the conditions per try.
Let's Begin!
Task description: You should use function

Посмотрим на то, как маскируется трейн

In [21]:
from toolbench.train.train import preprocess
from transformers import AutoTokenizer 

model_path = '/data6/ilseyar/toolbench/ToolLLaMA-2-7b'
tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        model_max_length=8192,
        padding_side="right",
        use_fast=False,
    )

In [22]:
test_output = preprocess([all_callable_examples[0]['conversations']], 
                         tokenizer, template=template)

In [36]:

labels = [label for label in test_output['labels'][0] if label != -100]
tokenizer.decode(labels)

'\nThought: \nAction: Finish\nAction Input: {"return_type": "give_answer", "final_answer": "The constructor standings after round 5 of the 2022 season are as follows:\\n1. Ferrari - 157 points, 2 wins\\n2. Red Bull - 151 points, 3 wins\\n...\\n\\nThe pitstop data for the race in 2021, round 10 are as follows:\\n- Driver: Leclerc, Stop: 1, Lap: 2, Time: 15:07:57, Duration: 34:07.731\\n- Driver: Hamilton, Stop: 1, Lap: 2, Time: 15:09:13, Duration: 26:17.540\\n..."}</s>'

In [38]:
tokenizer.decode(test_output['input_ids'][0])

'<s> System: You are AutoGPT, you can use many tools(functions) to do the following task.\nFirst I will give you the task description, and your task start.\nAt each step, you need to give your thought to analyze the status now and what to do next, with a function call to actually excute your step. Your output should follow this format:\nThought:\nAction\nAction Input:\n\nAfter the call, you will get the call result, and you are now in a new state.\nThen you will analyze your status now, then decide what to do next...\nAfter many (Thought-call) pairs, you finally perform the task, then you can give your finial answer.\nRemember: \n1.the state change is irreversible, you can\'t go back to one of the former state, if you want to restart the task, say "I give up and restart".\n2.All the thought is short, at most in 5 sentence.\n3.You can do more then one trys, so if your plan is to continusly try some conditions, you can do one of the conditions per try.\nLet\'s Begin!\nTask description: Y

In [35]:
test_output['labels'][0]

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 